# A4 成立域对照：开放连续谱（无回波）vs 闭合单模（周期回波）

**笔记**：notes/06_photon_topology/photon_first_principle_origin.md §3.7（A4 涌现不可逆推导候选，S4 可视化）

**论文**：paper/paper44_photon_topology.md §7.5 开放问题 7（A4 机制来源）

## 物理背景
A4 公理声称自发发射不可逆（σ_S3 仅 1→0）。本演示对比两种环境：

- **开放连续谱**（真空自由空间，Wigner–Weisskopf/Markov）：激发态存活概率单调指数衰减 $P_e(t)=e^{-\gamma_0 t}$——光子逃逸后**永不回波**（不可逆）；
- **闭合单模**（腔 QED 共振，Jaynes–Cummings）：$P_e(t)=\cos^2(gt)$ 周期振荡——激发在原子与腔模间**自发往返**（反向自发发生，可逆）。

**结论**：开放连续谱无回波 = 不可逆的定量表现；闭合系统回波 = **A4 的失效条件**（闭合系统 ⟹ 可逆，腔/镜 = 外部边界驱动）。

诚实边界：WW/JC 为标准量子光学事实；本演示为机制一致性的数值图示，非实验。

In [ ]:
# %matplotlib inline
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import os

# 中文字体回退（Windows）
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

FIG_DIR = os.path.normpath(os.path.join(os.getcwd(), "..", "figs"))
os.makedirs(FIG_DIR, exist_ok=True)


def ww_exact(gamma0, Gamma, t):
    """WW 精确解（洛伦兹谱 ⟹ 记忆核 K(τ)=(γ0Γ/2)e^{-Γτ}；
    ODE: c''+Γc'+(γ0Γ/2)c=0, c(0)=1, c'(0)=0）——非 Markov 参考"""
    D = Gamma**2 - 2.0 * gamma0 * Gamma
    if D > 0:
        s = np.sqrt(D)
        rp = (-Gamma + s) / 2
        rm = (-Gamma - s) / 2
        B = rp / (rp - rm)
        A = 1.0 - B
        return A * np.exp(rp * t) + B * np.exp(rm * t)
    if D == 0:
        return np.exp(-Gamma * t / 2) * (1 + (Gamma / 2) * t)
    s = np.sqrt(2 * gamma0 * Gamma - Gamma**2)
    return np.exp(-Gamma * t / 2) * (np.cos(s * t / 2) + (Gamma / s) * np.sin(s * t / 2))


In [ ]:
gamma0 = 1.0               # 物理衰减率（归一化）
t = np.linspace(0, 25, 2000)

# 开放连续谱：WW/Markov —— 存活概率单调指数衰减（无回波）
Pe_open = np.exp(-gamma0 * t)

# 闭合单模：JC 共振 |e,0> -> cos(gt)|e,0>，g=1（周期回波）
g_jc = 1.0
Pe_closed = np.cos(g_jc * t) ** 2

# 非 Markov 参考（γ0/Γ = 0.1 的有限带宽修正——与 Markov 指数偏差 ~O(γ0/Γ)）
Pe_nm = ww_exact(1.0, 10.0, t) ** 2

print(f"开放（连续谱 Markov）：P_e(25/γ) = {Pe_open[-1]:.2e}（单调 → 0，无回波）")
print(f"开放（连续谱 非 Markov γ/Γ=0.1）：P_e(25/γ) = {Pe_nm[-1]:.2e}（同样无回波）")
print(f"闭合（单模 JC）：P_e 周期回波 max = {Pe_closed.max():.3f}（反向自发 ⟹ 可逆）")

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 4.6))
ax.plot(t, Pe_open, lw=2.4, color="C0", label="Open continuum (WW): $P_e=e^{-\\gamma_0 t}$")
ax.plot(t, Pe_nm, lw=1.4, color="C1", ls="--", label="Open continuum (non-Markov, $\\gamma_0/\\Gamma=0.1$)")
ax.plot(t, Pe_closed, lw=1.6, color="C3", alpha=0.85, label="Closed single mode (JC): $P_e=\\cos^2(gt)$")
ax.annotate("NO REVIVAL\n(irreversible: field escapes)\n$P_e\\to e^{-\\gamma_0 t}\\to 0$",
            xy=(24, Pe_open[-1]), xytext=(12.5, 0.52),
            arrowprops=dict(arrowstyle="->", color="C0"), fontsize=9, color="C0")
ax.annotate("REVIVAL: $0\\to 1$ spontaneous\n$\\Rightarrow$ A4 failure condition:\nclosed $\\Rightarrow$ reversible",
            xy=(18.85, 1.0), xytext=(6.5, 0.72),
            arrowprops=dict(arrowstyle="->", color="C3"), fontsize=9, color="C3")
ax.axhline(0, color="k", lw=0.6)
ax.axvline(0, color="k", lw=0.6)
ax.set_xlabel("$t$ (units of $1/\\gamma_0$)")
ax.set_ylabel("$P_e(t)$")
ax.set_title("A4 domain contrast: open continuum (no revival) vs closed single mode (revival)")
ax.legend(fontsize=8.5, loc="center right")
fig.tight_layout()
fig_path = os.path.join(FIG_DIR, "photon_fig7_no_revival.png")
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"图已保存：{fig_path}")

## 解读

1. **开放连续谱（蓝实线）**：存活概率单调指数衰减 $e^{-\gamma_0 t}$，$t=25/\gamma_0$ 时降到 $\sim 10^{-11}$——光子逃逸进入连续谱后**永不回波**。非 Markov 修正（绿虚线，带宽有限）只改变衰减细节，同样无回波。
2. **闭合单模（红实线）**：$P_e=\cos^2(gt)$ 周期振荡——激发在原子与腔模间自发往返，**反向 $0\to1$ 自发发生**。
3. **这就是 A4 的失效条件**：A4 声称“反向不能自发”只在**开放连续谱**成立（成立域限定）；闭合系统（腔/镜 = 外部边界）⟹ 可逆。
4. **不可逆的机制**（三锚点）：发射选择推迟辐射条件（因果性）+ 自由带 a.c. 谱逃逸（RAGE/WW）+ 向内通道恒等于外部吸收（R 折叠）——见笔记 §3.7。

In [ ]:
# 光子逃逸：原子处场概率 e^{-γ0 t} -> 0；因果外向波包前沿 x = ct 向外移动
Gamma = gamma0 / 2.0                  # 自然线宽 = γ0/2
t2 = np.linspace(0, 10, 800)
p_atom = np.exp(-2 * Gamma * t2)      # |ψ(0,t)|² = e^{-γ0 t}

x = np.linspace(0, 8, 801)
T, X = np.meshgrid(t2, x)
# 因果外向波包：ψ(x,t) ∝ e^{-Γ(t-x/c)}·θ(t-x/c)，前沿 x=ct（推迟辐射条件）
packet = np.exp(-Gamma * (T - X)) * (T - X > 0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2))
ax1.semilogy(t2, p_atom, lw=2.2, color="C1")
ax1.set_ylim(1e-6, 1.1)
ax1.set_xlabel("$t$ (units of $1/\\gamma_0$)")
ax1.set_ylabel("$|\\psi(0,t)|^2$ at the atom")
ax1.set_title("Field at the atom empties: $e^{-\\gamma_0 t}\\to 0$")
ax1.axhline(1e-5, ls="--", color="gray", lw=0.8)
ax2.contourf(X, T, packet, levels=20, cmap="Blues")
ax2.plot(t2, t2, color="red", lw=2.2, label="packet front $x=ct$")
ax2.set_xlabel("$x$ (units of $c/\\gamma_0$)")
ax2.set_ylabel("$t$ (units of $1/\\gamma_0$)")
ax2.set_title("Causal outgoing wave packet (retarded light-cone front)")
ax2.legend(fontsize=9)
fig.tight_layout()
fig_path2 = os.path.join(FIG_DIR, "photon_fig8_packet_escape.png")
fig.savefig(fig_path2, dpi=150, bbox_inches="tight")
plt.show()
print(f"图已保存：{fig_path2}")

In [ ]:
print("=" * 70)
print("S4 可视化小结（A4 成立域对照）")
print(f"  开放连续谱：P_e(25/γ) = {Pe_open[-1]:.1e}（单调无回波 —— 不可逆的定量表现）")
print(f"  闭合单模：P_e 周期回波 max = {Pe_closed.max():.3f}（反向自发 ⟹ A4 失效条件）")
print(f"  原子处场概率：|ψ(0,10/γ)|² = {p_atom[-1]:.2e}（光子已逃逸，时间尺度 1/γ）")
print("  诚实边界：WW/JC 为标准量子光学事实；无回波 = 开放系统不可逆的数值演示，非实验。")
print(f"  图：{os.path.join(FIG_DIR, 'photon_fig7_no_revival.png')}")
print(f"     {os.path.join(FIG_DIR, 'photon_fig8_packet_escape.png')}")